# Math Lab (Linear Algebra + Probability) — AI Lab Instructor Notebook

*Math for ML · Easy*

Consolidate Week 2 with derivations + short computational checks.

**Outcome.** Students can derive key results, verify with small code experiments, and communicate clearly under interview constraints.

6 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Math Lab — FAANG-Level Mixed Problems

This lab is a *problem set + mini-verification* notebook covering projection matrices, PSD checks, least squares, Bayes intuition, and PCA links.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

**Why this works.** # Math Lab — Instructor (Solutions + Rationale)

## Task 2: Construct projection and verify properties
*Section: Projection Matrix Properties*

## Problem 1 — Projection Matrix Properties

Let P be a projection matrix onto a subspace.
1) Show that P^2 = P (idempotent).
2) For orthogonal projection, show P = P^T.

### Task 1.1
Construct projection onto span(u) and verify both properties.

**Hints**
- P = u u^T / (u^T u)

**Explain:** What does idempotent mean geometrically for projections?

In [ ]:
u = rng.standard_normal(5)
P = np.outer(u, u) / (u @ u)
check('idempotent', np.allclose(P @ P, P, atol=1e-8))
check('symmetric', np.allclose(P, P.T, atol=1e-8))

# Rationale: orthogonal projection matrices are exactly symmetric idempotent operators.

In [ ]:
# Checks
check('idempotent', np.allclose(P @ P, P, atol=1e-8))
check('symmetric', np.allclose(P, P.T, atol=1e-8))

**Why this works.** orthogonal projection matrices are symmetric and idempotent.

## Task 3: Verify X^T X is PSD
*Section: PSD Matrix Check*

## Problem 2 — PSD Matrix Check

Show that for any matrix X, A = X^T X is positive semidefinite (PSD).

### Task 2.1
Sample random X, build A, and verify v^T A v >= 0 for many random vectors v.

**Hints**
- v^T X^T X v = ||Xv||^2 >= 0

**Explain:** Why does v^T X^T X v equal ||Xv||^2?

In [ ]:
X = rng.standard_normal((10, 4))
A = X.T @ X
min_val = np.inf
for _ in range(100):
    v = rng.standard_normal(4)
    val = float(v.T @ A @ v)
    min_val = min(min_val, val)
print('min quadratic form', min_val)
check('psd_nonneg', min_val >= -1e-8)

# Rationale: A = X^T X is PSD because every quadratic form is a squared norm.

In [ ]:
# Checks
check('psd_nonneg', min_val >= -1e-8)

**Why this works.** A = X^T X is PSD since v^TAv = ||Xv||^2.

## Task 4: Normal equation vs lstsq
*Section: Least Squares Derivation*

## Problem 3 — Least Squares Derivation

Derive the normal equations for minimizing ||Xw - y||^2.

### Task 3.1
Compare w_hat from solve vs np.linalg.lstsq.

**Hints**
- w = (X^T X)^{-1} X^T y

**Explain:** Why is solve/lstsq preferred over explicit matrix inverse?

In [ ]:
n, d = 200, 5
X = rng.standard_normal((n, d))
w_true = rng.standard_normal(d)
y = X @ w_true + 0.1 * rng.standard_normal(n)

w_hat = np.linalg.solve(X.T @ X, X.T @ y)
w_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)
check('close', np.allclose(w_hat, w_lstsq, atol=1e-6))

# Rationale: direct inverse is numerically less stable; solve/lstsq is preferred.

In [ ]:
# Checks
check('close', np.allclose(w_hat, w_lstsq, atol=1e-6))

**Why this works.** solve/lstsq are numerically safer than explicit inverse.

## Task 5: Analytic vs simulated P(D|+)
*Section: Bayes + Base Rate*

## Problem 4 — Bayes + Base Rate (Derivation)

Re-derive P(D|+) for the disease test scenario and explain the base-rate fallacy.

### Task 4.1
Compute analytic posterior and verify with simulation.

**Hints**
- Use Bayes theorem and then Monte Carlo simulation.

**Explain:** Why can P(D|+) still be low when test sensitivity is high?

In [ ]:
P_D = 0.01
P_pos_given_D = 0.99
P_pos_given_notD = 0.05
P_D_given_pos = (P_pos_given_D * P_D) / (P_pos_given_D * P_D + P_pos_given_notD * (1 - P_D))

N = 200000
disease = rng.random(N) < P_D
test_pos = np.empty(N, dtype=bool)
test_pos[disease] = rng.random(disease.sum()) < P_pos_given_D
test_pos[~disease] = rng.random((~disease).sum()) < P_pos_given_notD
est = disease[test_pos].mean()
print('analytic', P_D_given_pos, 'sim', est)
check('close', abs(est - P_D_given_pos) < 0.01)

# Rationale: low base rate dominates posterior even with good sensitivity/specificity.

In [ ]:
# Checks
check('close', abs(est - P_D_given_pos) < 0.01)

**Why this works.** base rate strongly affects posterior probability.

## Task 6: Covariance eigenvectors vs SVD directions
*Section: PCA Link*

## Problem 5 — PCA Link

Explain why PCA components are eigenvectors of the covariance matrix.

### Task 5.1
Compute covariance eigenvectors and compare with SVD directions.

**Hints**
- Center X; Cov = X^T X/(n-1); compare eigvecs with V from SVD.

**Explain:** Why do right singular vectors align with covariance eigenvectors?

In [ ]:
X = rng.standard_normal((500, 20))
Xc = X - X.mean(axis=0, keepdims=True)
Cov = (Xc.T @ Xc) / (Xc.shape[0] - 1)

eigvals, eigvecs = np.linalg.eigh(Cov)
idx = np.argsort(eigvals)[::-1]
eigvecs = eigvecs[:, idx]

U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
V = Vt.T

k = 5
C = np.abs(eigvecs[:, :k].T @ V[:, :k])
print('abs alignment matrix (should be near diagonal)', C)
check('alignment', np.all(np.max(C, axis=1) > 0.9))

# Rationale: covariance eigendecomposition and SVD of centered data recover the same principal directions.

In [ ]:
# Checks
check('alignment', np.all(np.max(C, axis=1) > 0.9))

**Why this works.** PCA directions equal eigenvectors of covariance, same as right singular vectors of centered X.